In [15]:

import h5py
import numpy as np
from utils import save, load
from library import *
import gc
from continuous_process_library_terms import *
from commons_identify_models import *
import copy
import jax
import jax.numpy as jnp
from jax_cfd.base import boundaries, grids, pressure, finite_differences as fd
import jax_cfd.base as cfd
import jax_cfd.base.grids as grids
import jax_cfd.spectral as spectral
import numpy as np
import xarray 
import xarray as xr
import dataclasses
import time
jax.config.update("jax_enable_x64", True)


In [16]:

f = h5py.File('snapshots/snapshots_s1.h5', 'r')
print(list(f['tasks'].keys()))
'''
f['tasks']['buoyancy'] = f['buoyancy'][901:]
f['tasks']['pressure'] = f['pressure'][901:]
f['tasks']['velocity'] = f['velocity'][901:]
f['tasks']['vorticity'] = f['vorticity'][901:]
'''
#print(f.type)

scales = f['scales']


f = f['tasks']
list(f.keys())



velocity = f['velocity']
velocity = velocity[:,:,:,128:256]
vorticity = f['vorticity']
vorticity = vorticity[:,:,128:256]
print(velocity.shape)
pressure_import = f['pressure']
pressure_import = pressure_import[:,:,128:256]
buoyancy = f['buoyancy']
buoyancy=buoyancy[:,:,128:256]
print(velocity)
print(velocity.shape)
print(np.linspace(0,1,384).shape)

['buoyancy', 'pressure', 'velocity', 'vorticity']
(200, 2, 1024, 128)
[[[[-0.11340494 -0.1116512  -0.10992144 ...  0.18710332  0.19003854
     0.19290354]
   [-0.11149034 -0.1097677  -0.10806839 ...  0.18407345  0.18693989
     0.18973326]
   [-0.10961598 -0.10792684 -0.1062602  ...  0.18093738  0.18373282
     0.18645249]
   ...
   [-0.11941361 -0.11757813 -0.11576748 ...  0.19558617  0.19871262
     0.20177808]
   [-0.11736496 -0.11555509 -0.11376995 ...  0.19285638  0.19592176
     0.198923  ]
   [-0.11536248 -0.11357977 -0.1118215  ...  0.1900299   0.19303149
     0.19596585]]

  [[-0.30910752 -0.3108575  -0.31258834 ... -0.24436736 -0.24161103
    -0.2388044 ]
   [-0.31490067 -0.31661159 -0.3183012  ... -0.24473667 -0.24188336
    -0.23897813]
   [-0.32085513 -0.3225297  -0.32418055 ... -0.24497518 -0.24202241
    -0.23901635]
   ...
   [-0.2927075  -0.29458402 -0.29644637 ... -0.24252832 -0.24004576
    -0.23751876]
   [-0.29801185 -0.29984528 -0.30166313 ... -0.24325753 -0.24068

In [17]:
size_x = 1024
size_y = 128 
inner_steps = 1
outer_steps = 200

Lx =  4
Ly = 1/3
# Define the physical dimensions of the simulation.
Nx = size_x
Ny = size_y
Nt = outer_steps
dx = Lx/size_x
dy = Ly/size_y
print(dx)
Lt = 1.99

dt = 1e-2
print(dt)

Rayleigh = 1e6
Prandtl = 1
kappa = (Rayleigh * Prandtl)**(-1/2)
kappa

0.00390625
0.01


0.001

In [18]:
P = np.array(pressure_import)
B = np.array(buoyancy)

velocity = np.array(velocity) 
u_trajectory = velocity[:,0,:,:] #128:384
v_trajectory = velocity[:,1,:,:]
P = P[:,:,:]
B = B[:,:,:]
U = np.transpose(u_trajectory, (1,2,0))
V = np.transpose(v_trajectory, (1,2,0))
P = np.transpose(P, (1,2,0))
B = np.transpose(B, (1,2,0))
W = np.transpose(np.array(vorticity), (1,2,0))
temp = U
U = V
V = temp

x = scales['x_hash_89db91f00173784cfec0de1995a005933e22284d'][:]#['1.0'][:]    # If your x-basis is named 'x'
y = scales['z_hash_3c859c36a9c2f295aaab1dec0ec24d51221a3714'][128:256]#['1.0'][:]    # Likewise for y
t = scales['sim_time'][:]#['1.0'][:]
del(temp)
del(u_trajectory)
del(v_trajectory)
del(velocity)
del(pressure_import)
del(buoyancy)
del(f)
del(scales)

In [19]:
#INTERPOLATION
from scipy.interpolate import Akima1DInterpolator

print(x.shape, y.shape)
print(np.all(np.diff(y) > 0))
X, Y, T = np.meshgrid(x, y, t, indexing='ij')
ys = np.linspace(0, 1, 384)[128:256]
print(ys.shape)
u_lin = Akima1DInterpolator(y, U, axis=1, method='makima')(ys)
print("U_succ")
v_lin = Akima1DInterpolator(y, V, axis=1, method='makima')(ys)
b_lin = Akima1DInterpolator(y, B, axis=1, method='makima')(ys)
p_lin = Akima1DInterpolator(y, P, axis=1, method='makima')(ys)
w_lin = Akima1DInterpolator(y, W, axis=1, method='makima')(ys)
del(U)
del(V)
del(P)
del(B)
print(v_lin)
print(w_lin)

(1024,) (128,)
True
(128,)
U_succ
[[[-0.0777167  -0.07796488 -0.07821888 ... -0.10837009 -0.10791871
   -0.10746179]
  [-0.07664864 -0.07689923 -0.07715559 ... -0.10617872 -0.10572586
   -0.10526771]
  [-0.07557861 -0.0758315  -0.07609011 ... -0.10399907 -0.10354503
   -0.10308594]
  ...
  [ 0.11684004  0.11697216  0.11710662 ...  0.16359291  0.16376143
    0.16392784]
  [ 0.11905514  0.11919286  0.11933294 ...  0.16635987  0.16652819
    0.16669439]
  [ 0.12128102  0.12142429  0.12156996 ...  0.1691393   0.16930749
    0.16947353]]

 [[-0.07650662 -0.07675641 -0.07701229 ... -0.10715459 -0.1066976
   -0.10623492]
  [-0.07546438 -0.07571674 -0.07597513 ... -0.10497399 -0.10451544
   -0.10405147]
  [-0.07442049 -0.0746753  -0.07493607 ... -0.10280528 -0.10234547
   -0.10188049]
  ...
  [ 0.11474528  0.11488053  0.1150182  ...  0.16236366  0.16253565
    0.16270554]
  [ 0.11694095  0.11708175  0.11722498 ...  0.16510145  0.16527325
    0.16544294]
  [ 0.1191471   0.11929341  0.11944216 .

In [20]:
'''print(v_lin.shape)
v_lin=v_lin[5:1019,5:379,:]
u_lin=u_lin[5:1019,5:379,:]
b_lin=b_lin[5:1019,5:379,:]
p_lin=p_lin[5:1019,5:379,:]
w_lin=w_lin[5:1019,5:379,:]
print(v_lin.shape)'''

'print(v_lin.shape)\nv_lin=v_lin[5:1019,5:379,:]\nu_lin=u_lin[5:1019,5:379,:]\nb_lin=b_lin[5:1019,5:379,:]\np_lin=p_lin[5:1019,5:379,:]\nw_lin=w_lin[5:1019,5:379,:]\nprint(v_lin.shape)'

In [48]:
#%%prun # profiling
#print(P.shape)
uobs = Observable(string='u', rank=1)
pobs = Observable(string='p', rank=0)
bobs = Observable(string='b', rank=0) #?
zobs = Observable(string='z̄', rank=1)
wobs = Observable(string='w', rank=0)
observables = [uobs, bobs, pobs, zobs, wobs]#bobszobs

u = np.concatenate([v_lin[:, :, :, np.newaxis], u_lin[:, :, :, np.newaxis]], axis=3)
z = np.full(b_lin.shape, 1)
x = np.full(b_lin.shape, 0)
unit_vect = np.concatenate([z[:, :, :, np.newaxis], x[:, :, :, np.newaxis]], axis=3)
data_dict = {'u' : u, 'b' : b_lin, 'p' : p_lin, 'z̄' : unit_vect, 'w' : w_lin}
print(u.shape)
# fix random seed
np.random.seed(1)

world_size = np.array(p_lin.shape)
pad = 0

# fix random seed
np.random.seed(1)

dxs = [dx, dy, dt]

# initial setup of dataset
srd = SRDataset(world_size=world_size, data_dict=data_dict, observables=observables, dxs=dxs, 
                irreps=SRDataset.all_rank2_irreps())
                #irreps=(0, 1, 2))

# initialize libraries, domains, and weights
#srd.make_libraries(max_complexity=4, max_observables=3)
srd.make_libraries(max_complexity=3)
for irrep in srd.irreps:
    print(irrep, ":", len(srd.libs[irrep].terms))

import os
 
# Logical cores (including hyperthreading)
num_cores = os.cpu_count()
print(f"Logical CPU cores available: {num_cores}")
Ndomainz = len(srd.libs[srd.irreps[1]].terms)*10
num_processorz = num_cores - 4
Ndomainz_rounded = num_processorz * round(Ndomainz / num_processorz)
print(len(srd.libs[srd.irreps[1]].terms))
print(num_processorz)
print(Ndomainz_rounded)
dom_width = 32
dom_time = 25 #previously 20 (without interpolation)
pad = 0
srd.make_domains(ndomains=Ndomainz_rounded, domain_size=[dom_width, dom_width, dom_time], pad=pad)
srd.make_weights(m=12, qmax=0)
srd.set_LT_scale(L=dx*dom_width, T=dt*dom_time) # note that this line must go before make_library_matrices
srd.make_library_matrices(debug=False, parallel=True, num_processors=num_processorz)  # or whatever number of cores you want
'''
dom_width = 64 #previously 64 - smaller world size -> smaller domain??
dom_time = 20 #previously 20 (without interpolation)
print(len(srd.libs[srd.irreps[3]].terms))
srd.make_domains(ndomains=730, domain_size=[dom_width, dom_width, dom_time], pad=pad)
#srd.make_domains(ndomains=10, domain_size=[dom_width, dom_width, dom_time], pad=pad)
srd.make_weights(m=12, qmax=0)


print(Lt)
L = dom_width
T = dom_time
#print(NLt)
print(L, T)

srd.set_LT_scale(L, T) #T=1 # note that this line must go before make_library_matrices
srd.make_library_matrices(debug=False)'''

(1024, 128, 200, 2)
Rank 0 : 69
Rank 1 : 73
Antisymmetric rank 2 : 24
Symmetric trace-free rank 2 : 35
Logical CPU cores available: 128
73
124
744


'\ndom_width = 64 #previously 64 - smaller world size -> smaller domain??\ndom_time = 20 #previously 20 (without interpolation)\nprint(len(srd.libs[srd.irreps[3]].terms))\nsrd.make_domains(ndomains=730, domain_size=[dom_width, dom_width, dom_time], pad=pad)\n#srd.make_domains(ndomains=10, domain_size=[dom_width, dom_width, dom_time], pad=pad)\nsrd.make_weights(m=12, qmax=0)\n\n\nprint(Lt)\nL = dom_width\nT = dom_time\n#print(NLt)\nprint(L, T)\n\nsrd.set_LT_scale(L, T) #T=1 # note that this line must go before make_library_matrices\nsrd.make_library_matrices(debug=False)'

In [22]:
'''import numpy as np
import matplotlib.pyplot as plt
import os
import imageio.v2 as imageio

# === Inputs ===
dt = 1e-2
save_folder = "u_term_frames"
make_gif = True

# Create output folder
os.makedirs(save_folder, exist_ok=True)

# Transpose u and w to (t, x, y[, ...])
u_plot = np.transpose(u, (2, 0, 1, 3))  # from (Nx, Ny, Nt, 2) -> (Nt, Nx, Ny, 2)
w_plot = np.transpose(w_lin, (2, 0, 1))     # from (Nx, Ny, Nt) -> (Nt, Nx, Ny)
n_iter = u_plot.shape[0]
#print(u_plot.shape)
# === 1. Time derivative du/dt ===
du_dt = np.zeros_like(u_plot)
du_dt[1:-1] = (u_plot[2:] - u_plot[:-2]) / (2 * dt)
du_dt[0] = (u_plot[1] - u_plot[0]) / dt
du_dt[-1] = (u_plot[-1] - u_plot[-2]) / dt

# === 2. Convective term u_beta * ∂_beta u_alpha ===
def compute_convective_term(u_t):
    uy, ux = np.gradient(u_t[..., 0], axis=(0, 1))  # ∂y u_x, ∂x u_x
    vy, vx = np.gradient(u_t[..., 1], axis=(0, 1))  # ∂y u_y, ∂x u_y

    u_x = u_t[..., 0]
    u_y = u_t[..., 1]

    adv_x = u_x * ux + u_y * uy
    adv_y = u_x * vx + u_y * vy

    return np.stack([adv_x, adv_y], axis=-1)

u_adv = np.array([compute_convective_term(u_t) for u_t in u_plot])

# === 3. Plot frames ===
filenames = []

for i in range(n_iter):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    
    # Compute norms
    #print(i)
    norm_dt = np.linalg.norm(du_dt[i], axis=-1)
    norm_adv = np.linalg.norm(u_adv[i], axis=-1)
    w_i = w_plot[i]

    # Plot ∂t u
    im0 = axs[0].imshow(norm_dt.T, cmap='viridis', origin='lower')
    axs[0].set_title(rf"$\partial_t u$ at $t = {i * dt:.2f}$")
    plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)
    
    # Plot uβ ∂β uα
    im1 = axs[1].imshow(norm_adv.T, cmap='plasma', origin='lower')
    axs[1].set_title(r"$u_\beta \partial_\beta u_\alpha$")
    plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)
    
    # Plot scalar field w
    im2 = axs[2].imshow(w_i.T, cmap='coolwarm', origin='lower')
    axs[2].set_title(r"$w$ scalar field")
    plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

    for ax in axs:
        ax.axis('off')

    filename = os.path.join(save_folder, f"frame_{i:03d}.png")
    filenames.append(filename)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

# === 4. Optional GIF ===
if make_gif:
    gif_path = os.path.join(save_folder, "u_terms_with_w.gif")
    with imageio.get_writer(gif_path, mode='I', duration=0.05) as writer:
        for filename in filenames:
            image = imageio.imread(filename)
            writer.append_data(image)
    print(f"GIF saved to {gif_path}")
else:
    print(f"{n_iter} frames saved to '{save_folder}'")'''

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_3168316/2041005444.py:1: SyntaxWarning: invalid escape sequence '\p'
  '''import numpy as np


'import numpy as np\nimport matplotlib.pyplot as plt\nimport os\nimport imageio.v2 as imageio\n\n# === Inputs ===\ndt = 1e-2\nsave_folder = "u_term_frames"\nmake_gif = True\n\n# Create output folder\nos.makedirs(save_folder, exist_ok=True)\n\n# Transpose u and w to (t, x, y[, ...])\nu_plot = np.transpose(u, (2, 0, 1, 3))  # from (Nx, Ny, Nt, 2) -> (Nt, Nx, Ny, 2)\nw_plot = np.transpose(w_lin, (2, 0, 1))     # from (Nx, Ny, Nt) -> (Nt, Nx, Ny)\nn_iter = u_plot.shape[0]\n#print(u_plot.shape)\n# === 1. Time derivative du/dt ===\ndu_dt = np.zeros_like(u_plot)\ndu_dt[1:-1] = (u_plot[2:] - u_plot[:-2]) / (2 * dt)\ndu_dt[0] = (u_plot[1] - u_plot[0]) / dt\ndu_dt[-1] = (u_plot[-1] - u_plot[-2]) / dt\n\n# === 2. Convective term u_beta * ∂_beta u_alpha ===\ndef compute_convective_term(u_t):\n    uy, ux = np.gradient(u_t[..., 0], axis=(0, 1))  # ∂y u_x, ∂x u_x\n    vy, vx = np.gradient(u_t[..., 1], axis=(0, 1))  # ∂y u_y, ∂x u_y\n\n    u_x = u_t[..., 0]\n    u_y = u_t[..., 1]\n\n    adv_x = u_x * 

In [23]:
from commons_utils import *

lib1 = srd.libs[srd.irreps[1]]
for match in regex_find(lib1.terms, r'∂t u_α'):
    print(match)
dtu = lib1.Q[:, match[0]]

for match in regex_find(lib1.terms, r'u_β · ∂β u_α'):
    print(match)
adv = lib1.Q[:, match[0]]

for match in regex_find(lib1.terms, r'∂α p'):
    print(match)
dp = lib1.Q[:, match[0]]

for match in regex_find(lib1.terms, r'∂β² u_α'):
    print(match)
viscosity = 1e-3*lib1.Q[:, match[0]] #0.1

print(np.linalg.norm(dtu), np.linalg.norm(adv), np.linalg.norm(dp), np.linalg.norm(dtu+dp+adv-viscosity), 
      np.linalg.norm(dp-viscosity), np.linalg.norm(dtu+adv))

(54, ∂t u_α)
(39, u_β · ∂β u_α)
(29, ∂α p)
(53, ∂β² u_α)
1120.3763154741314 6031.015346737424 16018.292976486131 15629.451682318102 16014.768561855715 6044.968640222223


In [24]:
lib0 = srd.libs[srd.irreps[0]]
for match in regex_find(lib0.terms, r'∂α u_α'):
    print(match)
div = lib0.Q[:, match[0]]

print(np.linalg.norm(div))

(51, ∂α u_α)
19.092247727517773


In [25]:
'''from commons_identify_models import *
import copy
#Threshold: 1e-06, Initializer: combinatorial, Residual: matrix_relative, Threshold Type: information
libs = srd.libs

reg_opts_list = []
for irrep in srd.irreps:
    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold
    scaler = Scaler(sub_inds=None, char_sizes=libs[irrep].col_weights, row_norms=None)
    init = Initializer(method='combinatorial', start_k=2)
    #init = Initializer(method='combinatorial', start_k=9999)
    #init = Initializer(method='power', start_k=10)
    #res = Residual(residual_type='fixed_column', anchor_col=0)
    res = Residual(residual_type='matrix_relative')
    
    iterator = ModelIterator(max_k=10, backward_forward=True, max_passes=10)
    thres = Threshold(threshold_type='information', gamma=8, delta=1e-6, n_terms=None) #gamma-delta matter a lot
    #thres = Threshold(threshold_type='information', ic=AIC)
    #thres = Threshold(threshold_type='jump', gamma=1.5, n_terms=3)
    
    opts = {'scaler': scaler, 'initializer': init, 'residual': res,
            'model_iterator': iterator, 'threshold': thres}
    opts['verbose'] = False
    opts['inhomog'] = False
    opts['inhomog_col'] = None
    reg_opts_list.append(opts)

eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify([libs[i] for i in srd.irreps], 
reg_opts_list, threshold=1e-6, experimental=True) #threshold matters
#, max_equations=10)
print(derived_eqs)
print([reg_result.all_lambdas for reg_result in reg_results])'''

"from commons_identify_models import *\nimport copy\n#Threshold: 1e-06, Initializer: combinatorial, Residual: matrix_relative, Threshold Type: information\nlibs = srd.libs\n\nreg_opts_list = []\nfor irrep in srd.irreps:\n    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold\n    scaler = Scaler(sub_inds=None, char_sizes=libs[irrep].col_weights, row_norms=None)\n    init = Initializer(method='combinatorial', start_k=2)\n    #init = Initializer(method='combinatorial', start_k=9999)\n    #init = Initializer(method='power', start_k=10)\n    #res = Residual(residual_type='fixed_column', anchor_col=0)\n    res = Residual(residual_type='matrix_relative')\n\n    iterator = ModelIterator(max_k=10, backward_forward=True, max_passes=10)\n    thres = Threshold(threshold_type='information', gamma=8, delta=1e-6, n_terms=None) #gamma-delta matter a lot\n    #thres = Threshold(threshold_type='information', ic=AIC)\n    #thres = Threshold(threshold_type='jump

In [26]:
'''from commons_identify_models import *
import copy
 
libs = srd.libs
 
# Parameter ranges for different aspects of the regression process
threshold_values = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]
initializer_methods = ['combinatorial'] #, 'power'] 
residual_types = ['matrix_relative'] #, 'dominant_balance']
threshold_types = ['information']  #, 'jump'] # Added 'information' type
 
# Iterate through all combinations
for mk in range(5, 40):
    print("###############################################################k = " + str(mk) + "########################################")
    for threshold_loop in threshold_values:
        for init_method in initializer_methods:
            for res_type in residual_types:
                for thres_type in threshold_types:
                    print("-----------------------------------------------------------")
                    print(f"Threshold: {threshold_loop}, Initializer: {init_method}, Residual: {res_type}, Threshold Type: {thres_type}")
                    reg_opts_list = []
                    for irrep in srd.irreps:
                        scaler = Scaler(sub_inds=None, char_sizes=libs[irrep].col_weights, row_norms=None, unit_rows=True, train_fraction=0.75)
                        init = Initializer(method=init_method, start_k=mk)
                        if res_type == "dominant_balance":
                            res = Residual(residual_type='dominant_balance')
                        else:
                            res = Residual(residual_type='matrix_relative')
     
                        iterator = ModelIterator(max_k=mk, backward_forward=False, max_passes=30)
                        if thres_type == "jump":
                            thres = Threshold(threshold_type='jump', gamma=2, delta=1e-8)
                        else:  # 'information'
                            thres = Threshold(threshold_type='information', ic=AIC)
     
                        opts = {
                            'scaler': scaler,
                            'initializer': init,
                            'residual': res,
                            'model_iterator': iterator,
                            'threshold': thres,
                            'verbose': False,
                            'inhomog': False,
                            'inhomog_col': None
                        }
                        reg_opts_list.append(opts)
                    try:
                        eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify(
                            [libs[i] for i in srd.irreps], reg_opts_list, threshold=threshold_loop, experimental=True)
                    except TypeError as e:
                        print(f"Encountered an error: {e}")'''

'from commons_identify_models import *\nimport copy\n\nlibs = srd.libs\n\n# Parameter ranges for different aspects of the regression process\nthreshold_values = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]\ninitializer_methods = [\'combinatorial\'] #, \'power\'] \nresidual_types = [\'matrix_relative\'] #, \'dominant_balance\']\nthreshold_types = [\'information\']  #, \'jump\'] # Added \'information\' type\n\n# Iterate through all combinations\nfor mk in range(5, 40):\n    print("###############################################################k = " + str(mk) + "########################################")\n    for threshold_loop in threshold_values:\n        for init_method in initializer_methods:\n            for res_type in residual_types:\n                for thres_type in threshold_types:\n                    print("-----------------------------------------------------------")\n                    print(f"Threshold: {threshold_loop}, Initializer: {init_method}, Residual: {res_type}

In [27]:
'''from commons_identify_models import *
import copy
 
libs = srd.libs
 
# Parameter ranges for different aspects of the regression process
threshold_values = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]
initializer_methods = ['combinatorial'] #  'power', 
residual_types = ['matrix_relative']
threshold_types = ['information'] # Added 'information' type , 'jump'
 
# Iterate through all combinations
for mk in range(5, 40):
    print("###############################################################k = " + str(mk) + "########################################")
    for threshold_loop in threshold_values:
        for init_method in initializer_methods:
            for res_type in residual_types:
                for thres_type in threshold_types:
                    print("-----------------------------------------------------------")
                    print(f"Threshold: {threshold_loop}, Initializer: {init_method}, Residual: {res_type}, Threshold Type: {thres_type}")
                    reg_opts_list = []
                    for irrep in srd.irreps:
                        scaler = Scaler(sub_inds=None, char_sizes=libs[irrep].col_weights, row_norms=None, unit_rows=True, train_fraction=0.75)
                        init = Initializer(method=init_method, start_k=mk)
                        if res_type == "dominant_balance":
                            res = Residual(residual_type='dominant_balance')
                        else:
                            res = Residual(residual_type='matrix_relative')
     
                        iterator = ModelIterator(max_k=mk, backward_forward=False, max_passes=30)
                        if thres_type == "jump":
                            thres = Threshold(threshold_type='jump', gamma=2, delta=1e-6)
                        else:  # 'information'
                            thres = Threshold(threshold_type='information', ic=AIC)
     
                        opts = {
                            'scaler': scaler,
                            'initializer': init,
                            'residual': res,
                            'model_iterator': iterator,
                            'threshold': thres,
                            'verbose': False,
                            'inhomog': False,
                            'inhomog_col': None
                        }
                        reg_opts_list.append(opts)
                    try:
                        eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify(
                            [libs[i] for i in srd.irreps], reg_opts_list, threshold=threshold_loop, experimental=True)
                    except TypeError as e:
                        print(f"Encountered an error: {e}")'''

'from commons_identify_models import *\nimport copy\n\nlibs = srd.libs\n\n# Parameter ranges for different aspects of the regression process\nthreshold_values = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]\ninitializer_methods = [\'combinatorial\'] #  \'power\', \nresidual_types = [\'matrix_relative\']\nthreshold_types = [\'information\'] # Added \'information\' type , \'jump\'\n\n# Iterate through all combinations\nfor mk in range(5, 40):\n    print("###############################################################k = " + str(mk) + "########################################")\n    for threshold_loop in threshold_values:\n        for init_method in initializer_methods:\n            for res_type in residual_types:\n                for thres_type in threshold_types:\n                    print("-----------------------------------------------------------")\n                    print(f"Threshold: {threshold_loop}, Initializer: {init_method}, Residual: {res_type}, Threshold Type: {thres_typ

In [28]:
''' Threshold: 0.0001, Initializer: power, Residual: matrix_relative, Threshold Type: information
not picking up above 1024
sze 1024 = train res 9.04e-05, test res 7.16e-05
sze 512  = train res 9.33e-05, test res 7.39e-05
sze 256 = train res 9.40e-05, test res 7.45e-05
sze 128 = train res 9.42e-05, test res 7.46e-05
sze 64 = train res 9.43e-05, test res 7.47e-05
sze 32 = train res 9.43e-05, test res 7.47e-05

????Identified model: 0.355 · ∂α b · u_α + ∂t b = 0 (order 3, train res 5.89e-06, test res 1.29e-05)
1.1 e-6 Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00585 · ∂α b · u_α + 0.000168 · ∂α² b + 0.00551 · ∂t b + ∂α u_α = 0 (order 3, train res 1.42e-07, test res 2.34e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00585 · ∂α b · u_α + 0.000168 · ∂α² b + 0.00551 · ∂t b + ∂α u_α = 0 (order 3, train res 1.42e-07, test res 2.34e-07)
Identified model: 0.00131 · b · ∂t b + -0.252 · b · ∂α u_α + 0.00485 · ∂α b · u_α + 0.000182 · ∂α² b + ∂α u_α = 0 (order 3, train res 1.53e-07, test res 2.98e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00476 · ∂α b · u_α + 0.000179 · ∂α² b + 0.00498 · ∂t b + ∂α u_α = 0 (order 3, train res 1.51e-07, test res 2.88e-07)

'''

' Threshold: 0.0001, Initializer: power, Residual: matrix_relative, Threshold Type: information\nnot picking up above 1024\nsze 1024 = train res 9.04e-05, test res 7.16e-05\nsze 512  = train res 9.33e-05, test res 7.39e-05\nsze 256 = train res 9.40e-05, test res 7.45e-05\nsze 128 = train res 9.42e-05, test res 7.46e-05\nsze 64 = train res 9.43e-05, test res 7.47e-05\nsze 32 = train res 9.43e-05, test res 7.47e-05\n\n????Identified model: 0.355 · ∂α b · u_α + ∂t b = 0 (order 3, train res 5.89e-06, test res 1.29e-05)\n1.1 e-6 Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)\nIdentified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)\nIdentified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2

In [43]:
lib3 = srd.libs[srd.irreps[1]]
for idx, term in enumerate(lib3.terms):
    if str(term) == "b · z̄_α": #∂t b "u_β · ∂β u_α"
        print(f"Found at index {idx}: {term}") 
print(lib3)

Found at index 11: b · z̄_α
LibraryData(terms=[b · b · u_α, b · b · z̄_α, b · ∂α b, b · p · u_α, b · p · z̄_α, b · ∂α p, b · u_α, b · u_α · w, b · ∂t u_α, b · w · z̄_α, b · ∂α w, b · z̄_α, b · ∂t z̄_α, ∂α b, ∂α b · p, ∂α b · w, ∂t b · u_α, ∂t b · z̄_α, ∂t ∂α b, p · p · u_α, p · p · z̄_α, p · ∂α p, p · u_α, p · u_α · w, p · ∂t u_α, p · w · z̄_α, p · ∂α w, p · z̄_α, p · ∂t z̄_α, ∂α p, ∂α p · w, ∂t p · u_α, ∂t p · z̄_α, ∂t ∂α p, u_α, u_α · u_β · u_β, u_β · u_β · z̄_α, u_α · u_β · z̄_β, u_β · ∂β u_α, u_β · ∂α u_β, u_α · ∂β u_β, u_α · w, u_α · w · w, u_α · ∂t w, u_β · z̄_α · z̄_β, u_α · z̄_β · z̄_β, u_β · ∂α z̄_β, u_α · ∂β z̄_β, u_β · ∂β z̄_α, ∂α u_β · z̄_β, ∂β u_α · z̄_β, ∂β u_β · z̄_α, ∂α ∂β u_β, ∂β² u_α, ∂t u_α, ∂t u_α · w, ∂t² u_α, w · w · z̄_α, w · ∂α w, w · z̄_α, w · ∂t z̄_α, ∂α w, ∂t w · z̄_α, ∂t ∂α w, z̄_α, z̄_α · z̄_β · z̄_β, z̄_β · ∂β z̄_α, z̄_β · ∂α z̄_β, z̄_α · ∂β z̄_β, ∂α ∂β z̄_β, ∂β² z̄_α, ∂t z̄_α, ∂t² z̄_α], irrep=Rank 1, Q=array([[-2.270e+00,  5.243e+01,  9.157e+01, ..., -3.

In [49]:
from commons_identify_models import *

import copy
 
libs = srd.libs

lib3 = libs[srd.irreps[0]]
 
for aaaa in range(1,6):

    max_kk = 5

    print("Number of Terms in inhomogenous regression: ", aaaa)

    reg_opts_list = []
 
    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold

    scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)

    #init = Initializer(method='combinatorial', start_k=2)

    #init = Initializer(method='combinatorial', start_k=9999)

    init = Initializer(method='combinatorial', start_k=max_kk)

    #res = Residual(residual_type='fixed_column', anchor_col=0)

    #res = Residual(residual_type='dominant_balance')

    res = Residual(residual_type='matrix_relative')
 
    iterator = ModelIterator(max_k=max_kk, backward_forward=True, max_passes=20)

    thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=aaaa)

    #thres = Threshold(threshold_type='information', ic=AIC)
 
    opts = {'scaler': scaler, 'initializer': init, 'residual': res,

            'model_iterator': iterator, 'threshold': thres}

    opts['verbose'] = False

    opts['inhomog'] = True

    opts['inhomog_col'] = 21 #25
 
    reg_result = sparse_reg_bf(lib3.Q, **opts)

    zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]

    eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])
 
    print(eqn, "; residual:", reg_result.lambd)

    print("")
 

Number of Terms in inhomogenous regression:  1
∂t b = 0 ; residual: 0.014515690919272583

Number of Terms in inhomogenous regression:  2
b · ∂t b + -0.4842409319733291 · ∂t b = 0 ; residual: 0.0012411650836580941

Number of Terms in inhomogenous regression:  3
0.997210876602439 · ∂α b · u_α + -0.0009972808585397186 · ∂α² b + ∂t b = 0 ; residual: 2.5981876669928502e-05

Number of Terms in inhomogenous regression:  4
0.9971642457766812 · ∂α b · u_α + -0.0009959583484796478 · ∂α² b + ∂t b + -5.390231510298179e-06 · ∂α w · z̄_α = 0 ; residual: 2.339821941661519e-05

Number of Terms in inhomogenous regression:  5
0.9969829314629423 · ∂α b · u_α + -0.000995260248503698 · ∂α² b + ∂t b + -0.0003522924239583861 · ∂t b · w + -5.938257405365707e-06 · ∂α w · z̄_α = 0 ; residual: 2.2117026601043124e-05



In [50]:
from commons_identify_models import *

import copy
 
libs = srd.libs

lib3 = libs[srd.irreps[1]]
 
for aaaa in range(1,6):

    max_kk = 5

    print("Number of Terms in inhomogenous regression: ", aaaa)

    reg_opts_list = []
 
    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold

    scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)

    #init = Initializer(method='combinatorial', start_k=2)

    #init = Initializer(method='combinatorial', start_k=9999)

    init = Initializer(method='combinatorial', start_k=max_kk)

    #res = Residual(residual_type='fixed_column', anchor_col=0)

    #res = Residual(residual_type='dominant_balance')

    res = Residual(residual_type='matrix_relative')
 
    iterator = ModelIterator(max_k=max_kk, backward_forward=True, max_passes=20)

    thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=aaaa)

    #thres = Threshold(threshold_type='information', ic=AIC)
 
    opts = {'scaler': scaler, 'initializer': init, 'residual': res,

            'model_iterator': iterator, 'threshold': thres}

    opts['verbose'] = False

    opts['inhomog'] = True

    opts['inhomog_col'] = 38 #25
 
    reg_result = sparse_reg_bf(lib3.Q, **opts)

    zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]

    eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])
 
    print(eqn, "; residual:", reg_result.lambd)

    print("")
    
'''
m=16 wid=32 t=20 0.0020376223580152813 · b · w · z̄_α + u_α · ∂β u_β + -0.0011664537149430037 · u_β · ∂α u_β + -0.0013929899096620094 · u_β · ∂β u_α + -0.0009889312888799 · w · z̄_α = 0 ; residual: 5.182324428182633e-05
m=16 wid=32 t=10 -1.2283724576156495e-12 · b · ∂t u_α + -6.702584575029802e-13 · ∂α p + -6.84563071504909e-13 · u_β · ∂β u_α + z̄_β · ∂α z̄_β + -4.895045000496682e-06 · ∂β² z̄_α = 0 ; residual: 9.703533522417526e-16
m=16 wid=64 t=10 0.0019031700165843342 · b · ∂α b + -0.0009097219188053712 · ∂α b + u_α · ∂β u_β + -0.0011204040585573048 · u_β · ∂α u_β + -0.0013993672782674105 · u_β · ∂β u_α = 0 ; residual: 7.203979782477684e-05


'''
 

Number of Terms in inhomogenous regression:  1
u_β · ∂α u_β = 0 ; residual: 0.09181676893547912

Number of Terms in inhomogenous regression:  2
u_β · ∂α u_β + 0.9483226297978988 · u_β · ∂β u_α = 0 ; residual: 0.05051985754022252

Number of Terms in inhomogenous regression:  3
u_α · ∂β u_β + -0.0011497358372490431 · u_β · ∂α u_β + -0.0013966452473411109 · u_β · ∂β u_α = 0 ; residual: 6.187271819119535e-05

Number of Terms in inhomogenous regression:  4
0.0024419183168705315 · ∂α b · p + u_α · ∂β u_β + -0.0011236249827931307 · u_β · ∂α u_β + -0.001407460530413717 · u_β · ∂β u_α = 0 ; residual: 5.80280040873026e-05

Number of Terms in inhomogenous regression:  5
0.0020105302614244706 · b · w · z̄_α + u_α · ∂β u_β + -0.0011613284599298725 · u_β · ∂α u_β + -0.001393667741427496 · u_β · ∂β u_α + -0.0009761378201445048 · w · z̄_α = 0 ; residual: 5.1492870177718924e-05



'\nm=16 wid=32 t=20 0.0020376223580152813 · b · w · z̄_α + u_α · ∂β u_β + -0.0011664537149430037 · u_β · ∂α u_β + -0.0013929899096620094 · u_β · ∂β u_α + -0.0009889312888799 · w · z̄_α = 0 ; residual: 5.182324428182633e-05\nm=16 wid=32 t=10 -1.2283724576156495e-12 · b · ∂t u_α + -6.702584575029802e-13 · ∂α p + -6.84563071504909e-13 · u_β · ∂β u_α + z̄_β · ∂α z̄_β + -4.895045000496682e-06 · ∂β² z̄_α = 0 ; residual: 9.703533522417526e-16\nm=16 wid=64 t=10 0.0019031700165843342 · b · ∂α b + -0.0009097219188053712 · ∂α b + u_α · ∂β u_β + -0.0011204040585573048 · u_β · ∂α u_β + -0.0013993672782674105 · u_β · ∂β u_α = 0 ; residual: 7.203979782477684e-05\n\n\n'

In [51]:
from commons_identify_models import *

import copy
 
libs = srd.libs

lib3 = libs[srd.irreps[1]]
 
for aaaa in range(1,6):

    max_kk = 5

    print("Number of Terms in inhomogenous regression: ", aaaa)

    reg_opts_list = []
 
    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold

    scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)

    #init = Initializer(method='combinatorial', start_k=2)

    #init = Initializer(method='combinatorial', start_k=9999)

    init = Initializer(method='combinatorial', start_k=max_kk)

    #res = Residual(residual_type='fixed_column', anchor_col=0)

    #res = Residual(residual_type='dominant_balance')

    res = Residual(residual_type='matrix_relative')
 
    iterator = ModelIterator(max_k=max_kk, backward_forward=True, max_passes=20)

    thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=aaaa)

    #thres = Threshold(threshold_type='information', ic=AIC)
 
    opts = {'scaler': scaler, 'initializer': init, 'residual': res,

            'model_iterator': iterator, 'threshold': thres}

    opts['verbose'] = False

    opts['inhomog'] = True

    opts['inhomog_col'] = 11 #25
 
    reg_result = sparse_reg_bf(lib3.Q, **opts)

    zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]

    eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])
 
    print(eqn, "; residual:", reg_result.lambd)

    print("")
    
'''
m=16 wid=32 t=20 0.0020376223580152813 · b · w · z̄_α + u_α · ∂β u_β + -0.0011664537149430037 · u_β · ∂α u_β + -0.0013929899096620094 · u_β · ∂β u_α + -0.0009889312888799 · w · z̄_α = 0 ; residual: 5.182324428182633e-05
m=16 wid=32 t=10 -1.2283724576156495e-12 · b · ∂t u_α + -6.702584575029802e-13 · ∂α p + -6.84563071504909e-13 · u_β · ∂β u_α + z̄_β · ∂α z̄_β + -4.895045000496682e-06 · ∂β² z̄_α = 0 ; residual: 9.703533522417526e-16
m=16 wid=64 t=10 0.0019031700165843342 · b · ∂α b + -0.0009097219188053712 · ∂α b + u_α · ∂β u_β + -0.0011204040585573048 · u_β · ∂α u_β + -0.0013993672782674105 · u_β · ∂β u_α = 0 ; residual: 7.203979782477684e-05


'''
 

Number of Terms in inhomogenous regression:  1
b · z̄_α = 0 ; residual: 0.1534405020343868

Number of Terms in inhomogenous regression:  2
7.63278252263381e-15 · b · z̄_α + b · ∂t z̄_α = 0 ; residual: 3.0922721288940576e-25

Number of Terms in inhomogenous regression:  3
7.632782522243043e-15 · b · z̄_α + b · ∂t z̄_α + -9.160053006003421e-23 · ∂t b · z̄_α = 0 ; residual: 1.8943062445498585e-26

Number of Terms in inhomogenous regression:  4
7.632782522238923e-15 · b · z̄_α + b · ∂t z̄_α + -9.172778598521088e-23 · ∂t b · z̄_α + -1.0262417886921775e-25 · ∂t w · z̄_α = 0 ; residual: 1.8548195017079936e-26

Number of Terms in inhomogenous regression:  5
7.632782522234367e-15 · b · z̄_α + b · ∂t z̄_α + -9.210635209802992e-23 · ∂t b · z̄_α + -1.0593581402804918e-25 · ∂t w · z̄_α + -3.1969069745854913e-27 · ∂t ∂α w = 0 ; residual: 1.8365201809967667e-26



'\nm=16 wid=32 t=20 0.0020376223580152813 · b · w · z̄_α + u_α · ∂β u_β + -0.0011664537149430037 · u_β · ∂α u_β + -0.0013929899096620094 · u_β · ∂β u_α + -0.0009889312888799 · w · z̄_α = 0 ; residual: 5.182324428182633e-05\nm=16 wid=32 t=10 -1.2283724576156495e-12 · b · ∂t u_α + -6.702584575029802e-13 · ∂α p + -6.84563071504909e-13 · u_β · ∂β u_α + z̄_β · ∂α z̄_β + -4.895045000496682e-06 · ∂β² z̄_α = 0 ; residual: 9.703533522417526e-16\nm=16 wid=64 t=10 0.0019031700165843342 · b · ∂α b + -0.0009097219188053712 · ∂α b + u_α · ∂β u_β + -0.0011204040585573048 · u_β · ∂α u_β + -0.0013993672782674105 · u_β · ∂β u_α = 0 ; residual: 7.203979782477684e-05\n\n\n'

In [ ]:

libs = srd.libs
print(libs)
lib3 = libs[srd.irreps[3]]
 
 
reg_opts_list = []
 
# for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold
scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)
#init = Initializer(method='combinatorial', start_k=2)
#init = Initializer(method='combinatorial', start_k=9999)
init = Initializer(method='combinatorial', start_k=max_kk)
#res = Residual(residual_type='fixed_column', anchor_col=0)
res = Residual(residual_type='matrix_relative')
 
iterator = ModelIterator(max_k=2, backward_forward=True, max_passes=20)
thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=2)
#thres = Threshold(threshold_type='information', ic=AIC)
 
opts = {'scaler': scaler, 'initializer': init, 'residual': res,
        'model_iterator': iterator, 'threshold': thres}
opts['verbose'] = False
opts['inhomog'] = True
opts['inhomog_col'] = 0#25
 
reg_result = sparse_reg_bf(lib3.Q, **opts)
zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]
eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])
 
print(eqn, "; residual:", reg_result.lambd)